In [4]:
import os
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
import shutil
import random

BASE_DIR = "../data/"
OUTPUT_DIR = "ctc_merged_dataset"
TRAIN_SPLIT = 0.8
MAX_IMAGES_PER_DATASET = 600
RANDOM_SEED = 42

# Dataset configurations
DATASETS = [
    #"BF-C2DL-HSC",
    #"BF-C2DL-MuSC",
    "DIC-C2DH-HeLa",
    #"Fluo-C2DL-Huh7",
    #"Fluo-C2DL-MSC",
    #"Fluo-N2DH-GOWT1",
    #"Fluo-N2DH-SIM",
    #"Fluo-N2DL-HeLa",
    #"PhC-C2DH-U373",
    #"PhC-C2DL-PSC"
]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)


def split_train_val(items, train_split, seed):
    items = list(items)
    random.Random(seed).shuffle(items)
    n_train = int(len(items) * train_split)
    return items[:n_train], items[n_train:]


def convert_dataset_to_jpg(dataset_path, dataset_name):
    tmp_root = f"{dataset_name}_tmp_yolo_train"
    converted = 0

    for subfolder in ["01", "02"]:
        src_dir = os.path.join(dataset_path, subfolder)
        if not os.path.exists(src_dir):
            print(f"Warning: {src_dir} not found")
            continue

        dst_dir = os.path.join(tmp_root, subfolder)
        os.makedirs(dst_dir, exist_ok=True)

        tif_files = sorted(Path(src_dir).glob("*.tif")) + sorted(Path(src_dir).glob("*.tiff"))
        for tif_path in tif_files:
            image = cv2.imread(str(tif_path), cv2.IMREAD_UNCHANGED)
            if image is None:
                print(f"Failed to read {tif_path.name}, skipping.")
                continue

            # Convert 16-bit (or other formats) to 8-bit
            if image.dtype != np.uint8:
                max_val = float(image.max())
                if max_val <= 0:
                    image = np.zeros_like(image, dtype=np.uint8)
                else:
                    image = cv2.convertScaleAbs(image, alpha=255.0 / max_val)
                    
            stem = tif_path.stem
            if stem.startswith("t"):
                stem = stem[1:]

            cv2.imwrite(os.path.join(dst_dir, f"{stem}.jpg"), image)
            converted += 1

    print(f"Converted {converted} images -> {tmp_root}/")
    return tmp_root


def get_bounding_boxes_from_label_mask(
    mask: np.ndarray,
    background: int = 0,
    connectivity: int = 1,
    min_area: int = 1,
    pad: int = 0
):
    """Extract bounding boxes from segmentation mask"""
    H, W = mask.shape
    labels = np.unique(mask)
    labels = labels[labels != background]

    boxes = []
    for lb in labels:
        binary = (mask == lb).astype(np.uint8)
        if binary.sum() < min_area:
            continue
        conn = 4 if connectivity == 1 else 8
        n_labels, comp = cv2.connectedComponents(binary, connectivity=conn)
        for ridx in range(1, n_labels):
            ys, xs = np.where(comp == ridx)
            if ys.size == 0:
                continue
            area = int(ys.size)
            if area < min_area:
                continue
            y1, x1 = ys.min(), xs.min()
            y2, x2 = ys.max() + 1, xs.max() + 1
            if pad:
                y1 = max(0, y1 - pad)
                x1 = max(0, x1 - pad)
                y2 = min(H, y2 + pad)
                x2 = min(W, x2 + pad)
            boxes.append([int(x1), int(y1), int(x2), int(y2)])
    return boxes


def mask_to_yolo_format(mask_path, img_width, img_height):
    """Convert segmentation mask to YOLO format annotations"""
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    if mask is None:
        return []

    boxes = get_bounding_boxes_from_label_mask(
        mask, background=0, connectivity=1, min_area=5, pad=0
    )

    annotations = []
    for box in boxes:
        x1, y1, x2, y2 = box

        w = x2 - x1
        h = y2 - y1

        # Convert to YOLO format (normalized)
        x_center = (x1 + w / 2) / img_width
        y_center = (y1 + h / 2) / img_height
        width = w / img_width
        height = h / img_height

        annotations.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return annotations


def find_images_in_dataset(dataset_path):
    """Find all images in subfolders 01 and 02, skipping dataset-specific images."""

    images = []

    for subfolder in ["01", "02"]:
        img_dir = os.path.join(dataset_path, subfolder)
        if not os.path.exists(img_dir):
            print(f"Warning: {img_dir} not found")
            continue

        for img_path in Path(img_dir).glob("*.jpg"):
            name = img_path.name

            if "Fluo-N2DL-HeLa" in dataset_path:
                if subfolder == "01" and name in [
                    "012.jpg", "015.jpg", "020.jpg", "021.jpg", "022.jpg", "023.jpg",
                    "025.jpg", "029.jpg", "038.jpg", "039.jpg", "040.jpg", "050.jpg",
                    "051.jpg", "053.jpg", "054.jpg", "078.jpg", "079.jpg", "080.jpg",
                    "081.jpg"
                ]:
                    continue
                if subfolder == "02" and name in [
                    "023.jpg", "035.jpg", "036.jpg", "067.jpg", "078.jpg", "087.jpg"
                ]:
                    continue

            elif "Fluo-N2DH-GOWT1" in dataset_path:
                if subfolder == "01" and name in [
                    "002.jpg", "005.jpg", "006.jpg", "007.jpg", "008.jpg", "010.jpg",
                    "012.jpg", "013.jpg", "014.jpg", "015.jpg", "016.jpg", "017.jpg",
                    "018.jpg", "020.jpg", "027.jpg", "028.jpg", "029.jpg", "032.jpg",
                    "034.jpg", "037.jpg", "040.jpg", "041.jpg", "044.jpg", "045.jpg",
                    "046.jpg", "047.jpg", "080.jpg", "091.jpg"
                ]:
                    continue

                if subfolder == "02" and name in [
                    "022.jpg", "025.jpg", "027.jpg", "028.jpg", "029.jpg", "030.jpg",
                    "035.jpg", "039.jpg", "046.jpg", "047.jpg", "048.jpg", "060.jpg",
                    "065.jpg", "076.jpg", "082.jpg"
                ]:
                    continue

            elif "Fluo-C2DL-MSC" in dataset_path:
                if subfolder == "01" and name in [
                    "009.jpg", "028.jpg", "036.jpg", "046.jpg", "047.jpg",
                    "030.jpg", "031.jpg"
                ]:
                    continue

            # Add to list if not skipped
            images.append((img_path, subfolder))

    return images


def get_mask_path(img_path, dataset_path, subfolder):
    """Get corresponding mask file path with priority search order."""

    img_name = img_path.stem  # e.g., '012'
    mask_file = f"man_seg{img_name}.tif"

    search_dirs = [
        os.path.join(dataset_path, f"{subfolder}_GT", "SEG"),
        os.path.join(dataset_path, f"{subfolder}_ST", "SEG"),
    ]

    for dir_path in search_dirs:
        candidate = os.path.join(dir_path, mask_file)
        if os.path.exists(candidate):
            return candidate

    return None


def process_dataset(dataset_name, train_files, val_files):
    """Process one dataset's train and val splits"""
    dataset_path = os.path.join(BASE_DIR, dataset_name)

    stats = {
        'train_success': 0,
        'train_fail': 0,
        'val_success': 0,
        'val_fail': 0
    }

    for img_path, subfolder in train_files:
        if process_single_image(img_path, dataset_path, 'train', dataset_name, subfolder):
            stats['train_success'] += 1
        else:
            stats['train_fail'] += 1

    for img_path, subfolder in val_files:
        if process_single_image(img_path, dataset_path, 'val', dataset_name, subfolder):
            stats['val_success'] += 1
        else:
            stats['val_fail'] += 1

    return stats


def process_single_image(img_path, dataset_path, split_name, dataset_name, subfolder):
    """Process a single image and its mask"""
    try:
        mask_path = get_mask_path(img_path, dataset_path, subfolder)

        if mask_path is None:
            print(f"Warning: Mask not found for {img_path} in {subfolder}_GT")
            return False

        img = Image.open(img_path)
        img_width, img_height = img.size

        annotations = mask_to_yolo_format(mask_path, img_width, img_height)

        if len(annotations) == 0:
            print(f"Warning: No objects found in {mask_path}")
            return False

        new_filename = f"{dataset_name}_{subfolder}_{img_path.name}"
        base_name = Path(new_filename).stem

        output_img_path = f"{OUTPUT_DIR}/images/{split_name}/{base_name}.jpg"
        if img_path.suffix.lower() in ['.tif', '.tiff']:
            img.convert('RGB').save(output_img_path, 'JPEG')
        else:
            shutil.copy(img_path, output_img_path)

        label_file = f"{OUTPUT_DIR}/labels/{split_name}/{base_name}.txt"
        with open(label_file, 'w') as f:
            f.write('\n'.join(annotations))

        return True

    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return False


print("Starting dataset processing...")
total_stats = {'train': 0, 'val': 0, 'failed': 0}

for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*60}")

    dataset_path = os.path.join(BASE_DIR, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"Warning: Dataset path not found: {dataset_path}")
        continue

    tmp_path = convert_dataset_to_jpg(dataset_path, dataset_name)

    all_images = find_images_in_dataset(tmp_path)
    print(f"Found {len(all_images)} total images")

    if len(all_images) > MAX_IMAGES_PER_DATASET:
        all_images = random.sample(all_images, MAX_IMAGES_PER_DATASET)
        print(f"Sampled down to {MAX_IMAGES_PER_DATASET} images")

    train_files, val_files = split_train_val(
        all_images, TRAIN_SPLIT, RANDOM_SEED
    )
    print(f"Split: {len(train_files)} train, {len(val_files)} val")

    stats = process_dataset(dataset_name, train_files, val_files)

    print(f"\nResults for {dataset_name}:")
    print(f"  Train: {stats['train_success']} success, {stats['train_fail']} failed")
    print(f"  Val: {stats['val_success']} success, {stats['val_fail']} failed")

    total_stats['train'] += stats['train_success']
    total_stats['val'] += stats['val_success']
    total_stats['failed'] += stats['train_fail'] + stats['val_fail']

    shutil.rmtree(tmp_path, ignore_errors=True)

print(f"\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")
print(f"Total train images: {total_stats['train']}")
print(f"Total val images: {total_stats['val']}")
print(f"Total failed: {total_stats['failed']}")
print(f"\nDataset ready at: {OUTPUT_DIR}")

yaml_content = f"""# Cell Detection Dataset
path: {os.path.abspath(OUTPUT_DIR)}
train: images/train
val: images/val

nc: 1  # number of classes
names: ['cell']  # class names
"""

with open(f"{OUTPUT_DIR}/data.yaml", 'w') as f:
    f.write(yaml_content)

print(f"Created data.yaml configuration file")

Starting dataset processing...

Processing: DIC-C2DH-HeLa
Converted 168 images -> DIC-C2DH-HeLa_tmp_yolo_train/
Found 168 total images
Split: 134 train, 34 val

Results for DIC-C2DH-HeLa:
  Train: 134 success, 0 failed
  Val: 34 success, 0 failed

FINAL SUMMARY
Total train images: 134
Total val images: 34
Total failed: 0

Dataset ready at: ctc_merged_dataset
Created data.yaml configuration file
